# Lab 4

## Introduction

Train an autoencoder using the MagnaTagATune (MTAT) dataset
1. Adapt the provided autoencoder implementation and train the model on MTAT
* You can reuse the data loading code from the previous assignment.
2. Decide how to deal with longer audios
* Suggested strategies:
  * Crop audio segments to few seconds
  * Reduce the number of layers or kernels in the conv layers
3. Try to improve the encoder reconstruction quality
* In terms of validation MSE minimization
* It is not allowed to modify the original bottleneck layer compression rate.
* Things to try:
  * More advances perceptual losses
  * Use more layers
4. Write a short report documenting steps 1, 2, and 3.
* The report should be included in the notebook
* The notebook should be self-explanatory, and executable (i.e., it should contain the code to install dependencies and
setup the dataset). The assignment will be entirely evaluated on the contents of the notebook.

## Imports

In [1]:
!pip install torchmetrics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 46.3 MB/s eta 0:00:00


In [2]:
import os
from google.colab import drive
from google.colab import files
import pandas as pd
import zipfile
import matplotlib.pyplot as plt
import numpy as np
import librosa.display
import librosa
import glob
import random
from IPython.display import Audio

# Import necessary libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

from tqdm import trange
from tqdm import tqdm, trange

from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader, Dataset
from torchaudio.transforms import MelSpectrogram, Resample
from tqdm.autonotebook import tqdm

# Metrics
import torchmetrics
from sklearn.metrics import classification_report

/tmp/ipython-input-2721372857.py:28: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [3]:
# Setup device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
import warnings

# For muting error
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning, module='librosa')

## Data Importing

In [5]:
if not os.path.isdir("/content/drive"):
  drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
FINAL_ZIP_NAME = 'magna_tag_atune_complete.zip'
DESTINATION_FOLDER = 'dataset'
BASE_PATH = '/content/drive/MyDrive/MLSM/Lab3' # Set this to the directory where your .001 files are located

with zipfile.ZipFile(os.path.join(BASE_PATH, FINAL_ZIP_NAME), 'r') as zf:
    zf.extractall(DESTINATION_FOLDER)

print(f"Success! All audio clips have been extracted to: {DESTINATION_FOLDER}")

Success! All audio clips have been extracted to: dataset


## Data Loader


The strategy of cropping audio segments is superior to reducing the number of layers or kernels in the convolutional layers, especially since the goal is to maximize reconstruction quality (minimizing the validation Mean Squared Error).

### Why Cropping is the Preferred Strategy

Maximizing Model Capacity for Quality

The primary reason for choosing cropping is that it allows you to maintain or even increase the model's capacity.

Reducing the input length (from 30 seconds to 3 seconds of raw audio) dramatically decreases the memory footprint required for each training example and the subsequent feature maps generated by the convolutional layers. This enables to use a deeper (more layers) and wider (more kernels/channels) network. A model with greater capacity is inherently better suited to learn the complex hierarchical features necessary to achieve high-fidelity reconstruction, directly addressing your core assignment requirement to improve quality.

Avoiding Detrimental Capacity Reduction

The alternative, reducing the number of layers or kernels, directly sacrifices the model's learning capacity. While this might allow the model to process the full 30-second audio, the resulting network is simpler and fundamentally less powerful. A less capable model will struggle to encode and decode such a large amount of musical information into your fixed bottleneck size, leading to a higher reconstruction error and failing to meet the goal of improving validation MSE.

In summary, cropping provides the necessary memory relief while allowing to fully utilize the computational complexity of your network, which is key to achieving the best possible reconstruction quality.

In [7]:
class MTATSegmentDataset(Dataset):
    """
    Dataset to load random 5-second segments of raw audio (1D) from MTAT,
    handling the train/val/test split internally based on folder names.
    """
    def __init__(self, base_dir, split, sample_rate=44100, duration_seconds=3):
        # Configuration for the split folders
        split_map = {
            'train': [str(i) for i in range(10)] + ['a', 'b'], # 0-9, a, b
            'val': ['c'],
            'test': ['d', 'e', 'f']
        }

        if split not in split_map:
            raise ValueError(f"Invalid split name: {split}. Must be 'train', 'val', or 'test'.")

        self.sample_rate = sample_rate
        self.segment_samples = sample_rate * duration_seconds
        self.audio_paths = self._collect_audio_paths(base_dir, split_map[split])

        print(f"Initialized {split} dataset with {len(self.audio_paths)} files.")

    def _collect_audio_paths(self, base_dir, target_folders):
        """Helper to find and collect all audio paths within the target folders."""
        audio_paths = []

        for folder_name in target_folders:
            folder_path = os.path.join(base_dir, folder_name)
            audio_paths.extend(glob.glob(os.path.join(folder_path, '**', '*.mp3'), recursive=True))

        return audio_paths

    def __len__(self):
        return len(self.audio_paths)

    def __getitem__(self, index):
        audio_path = self.audio_paths[index]
        segment = None

        # 1. Load the audio
        try:
            full_audio, sr = librosa.load(audio_path, sr=self.sample_rate, mono=True)
        except Exception as e:
            # Border Case Handling: return silence if file fails to load
            #print(f"Error loading {audio_path}: {e}. Returning zero segment.")
            segment = np.zeros(self.segment_samples, dtype=np.float32)
            full_audio = segment # Ensure full_audio exists for subsequent logic

        # 2. Random Cropping Logic
        if segment is None: # Only run if the file loaded successfully
            max_start_sample = len(full_audio) - self.segment_samples

            if max_start_sample <= 0:
                # Pad if audio is too short
                segment = librosa.util.fix_length(full_audio, size=self.segment_samples, mode='constant')
            else:
                # Choose a random start point and crop
                start_sample = np.random.randint(0, max_start_sample)
                end_sample = start_sample + self.segment_samples
                segment = full_audio[start_sample:end_sample]

        # 3. Conversion to Tensor [1, Length]
        segment_tensor = torch.tensor(segment).unsqueeze(0).float()

        return segment_tensor, audio_path

I've decided to use 3-second audio segments (splits) for the MagnaTagATune (MTAT) dataset, which is an increase from the 1-second segments used in the original GitHub example.

This choice represents a critical balance across three factors:

* Reconstruction Quality: Increasing the segment length from 1 second to 3 seconds provides the autoencoder with more temporal context per sample. This increased context allows the 1D convolutional layers to capture and encode more significant musical structures (e.g., rhythmic patterns, harmonic movement) into the bottleneck. This directly supports the primary goal of improving the autoencoder's reconstruction quality (minimizing the validation MSE).

* Memory Management: While a 5-second segment would offer even more context, 3 seconds is a safer compromise to prevent the CUDA out of memory errors encountered previously. Shorter segments require less GPU memory for the input data and the intermediate feature maps, keeping the training feasible.

* Execution Speed: Choosing 3 seconds ensures that the training execution time remains practical. Using the full 30-second audio files, or even very long segments, would drastically increase the time needed to complete each epoch.

In short, the 3-second segment length is an optimized hyperparameter choice designed to maximize the learning capacity of the autoencoder while maintaining computational efficiency and stability.

In [8]:
# --- Configuration ---
BASE_MTAT_DIR = "dataset"
BATCH_SIZE = 16
FS = 16000
TIME = 3
# ---------------------


I address the Out Of Memory (OOM) error solution by detailing the steps I took in the past tense, focusing on the rationale behind the memory management choices.

I began by reducing the audio segment duration before resorting to any other measure. I chose this because the input to the 1D raw audio autoencoder is exceptionally large—over 200k samples per segment. This temporal length directly determines the size of all feature maps generated by the deep convolutional layers. By shrinking the input dimension, I achieved a fundamental and comprehensive reduction in memory consumption across the entire model architecture.

Subsequently, if the memory issue persisted after the segment length adjustment, I proceeded to reduce the batch size. This was my secondary measure, implemented to limit the number of samples being processed concurrently.

This approach ensured I tackled the root cause of the high memory usage at the data level (segment length) before compromising the efficiency of parallel processing (batch size).

In [9]:
# Datasets
train_dataset = MTATSegmentDataset(base_dir=BASE_MTAT_DIR, split='train', sample_rate=FS, duration_seconds=TIME)
val_dataset = MTATSegmentDataset(base_dir=BASE_MTAT_DIR, split='val', sample_rate=FS, duration_seconds=TIME)
test_dataset = MTATSegmentDataset(base_dir=BASE_MTAT_DIR, split='test', sample_rate=FS, duration_seconds=TIME)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Initialized train dataset with 18709 files.
Initialized val dataset with 1825 files.
Initialized test dataset with 5329 files.


## Models

### Auxiliar Functions

In [10]:
def train_autoencoder(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
    n_epochs: int = 5,
    learning_rate: float = 1e-3,
    criterion: nn.Module = nn.MSELoss(),
    loss_function: callable = None  # Use this for multi_scale_stft_loss
) -> tuple[nn.Module, dict]:
    """
    Trains and validates an AudioAutoencoder using a standard training loop.

    Args:
        model (nn.Module): The initialized AudioAutoencoder model instance.
        train_loader (DataLoader): DataLoader for the training set.
        val_loader (DataLoader): DataLoader for the validation set.
        device (torch.device): Device ('cuda' or 'cpu') for training.
        n_epochs (int): Number of epochs to train.
        learning_rate (float): Learning rate for the Adam optimizer.
        criterion (nn.Module): Standard loss function (MSELoss by default).
        loss_function (callable): The primary loss function to use (e.g.,
                                  multi_scale_stft_loss). If None, uses criterion.

    Returns:
        tuple[nn.Module, dict]: The trained model and a dictionary with the loss history.
    """
    # Use the specific loss function if provided (e.g., STFT loss); otherwise, use the standard criterion (MSE).
    current_loss_func = loss_function if loss_function is not None else criterion

    # Initialize the optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    # Move the model to the specified device
    model.to(device)

    history = {'train_loss': [], 'val_loss': []}

    pbar = trange(n_epochs, desc="Training", unit="epoch")
    for epoch in pbar:
        # ---------------------
        # 1. Training Loop
        # ---------------------
        model.train()
        loss_train = []

        # Note: Iterating over (X, _) as the autoencoder target Y is X (the input)
        for X, _ in train_loader:
            x = X.to(device)
            optimizer.zero_grad()

            x_reconstr, _ = model(x)

            # Calculate loss using the specified function
            loss = current_loss_func(x, x_reconstr)

            loss.backward()
            optimizer.step()
            loss_train.append(loss.item())

        loss_train = np.mean(loss_train)
        history['train_loss'].append(loss_train)

        # ---------------------
        # 2. Validation Loop
        # ---------------------
        model.eval()
        loss_val = []

        with torch.no_grad(): # Disable gradient calculation for validation
            for X, _ in val_loader:
                x = X.to(device)

                x_reconstr, _ = model(x)

                # Calculate loss
                loss = current_loss_func(x, x_reconstr)

                loss_val.append(loss.item())

        loss_val = np.mean(loss_val)
        history['val_loss'].append(loss_val)

        # Update progress bar
        pbar.set_postfix({
            "epoch": epoch+1,
            "train loss": f"{loss_train:.5f}",
            "val loss": f"{loss_val:.5f}"
        })

    return history

### Arquitectures

https://github.com/MTG/machine_learning_for_sound_and_music_25/blob/main/notebooks/06_01_autoencoders.ipynb

In [11]:
class AudioAutoencoder(nn.Module):
    """A more capable audio autoencoder featuring batch-norm, PReLu activations,
    and sharedd indices for the downsampling and upsampling layers.
    """
    def __init__(
        self,
        input_length=16000,
        bottleneck_size=64,
        kernel_size=5,
        channels=8,
        channel_mult=2,
        downsample=4,
    ):
        super().__init__()
        assert kernel_size % 2 == 1, "kernel_size should be odd for same padding"

        padding = kernel_size // 2

        # ----- Encoder -----
        self.enc_conv1 = nn.Conv1d(1, channels, kernel_size=kernel_size, padding=padding)
        self.enc_norm1 = nn.BatchNorm1d(channels)
        self.enc_act1 = nn.PReLU()
        self.pool1 = nn.MaxPool1d(2, stride=downsample, return_indices=True)

        self.enc_conv2 = nn.Conv1d(channels, channels * channel_mult, kernel_size=kernel_size, padding=padding)
        self.enc_norm2 = nn.BatchNorm1d(channels * channel_mult)
        self.enc_act2 = nn.PReLU()
        self.pool2 = nn.MaxPool1d(2, stride=downsample, return_indices=True)

        self.enc_conv3 = nn.Conv1d(channels * channel_mult, channels * channel_mult ** 2, kernel_size=kernel_size, padding=padding)
        self.enc_norm3 = nn.BatchNorm1d(channels * channel_mult ** 2)
        self.enc_act3 = nn.PReLU()
        self.pool3 = nn.MaxPool1d(2, stride=downsample, return_indices=True)

        enc_out_len = input_length // downsample ** 3 * channels * channel_mult ** 2  # 3× pooling halves length thrice
        self.enc_fc = nn.Linear(enc_out_len, bottleneck_size)

        comp_rate =  input_length / bottleneck_size
        print(f"Compression rate: {comp_rate:.1f}X")

        # ----- Decoder -----
        self.dec_fc = nn.Linear(bottleneck_size, enc_out_len)

        self.unpool3 = nn.MaxUnpool1d(2, stride=downsample)
        self.dec_deconv3 = nn.Conv1d(channels * channel_mult ** 2, channels * channel_mult, kernel_size=kernel_size, padding=padding)
        self.dec_act3 = nn.PReLU()

        self.unpool2 = nn.MaxUnpool1d(2, stride=downsample)
        self.dec_deconv2 = nn.Conv1d(channels * channel_mult, channels, kernel_size=kernel_size, padding=padding)
        self.dec_act2 = nn.PReLU()

        self.unpool1 = nn.MaxUnpool1d(2, stride=downsample)
        self.dec_deconv1 = nn.Conv1d(channels, 1, kernel_size=kernel_size, padding=padding)
        self.dec_tanh = nn.Tanh()

    def encoder(self, x):
        sizes, indices = [], []

        x = self.enc_act1(self.enc_norm1(self.enc_conv1(x)))
        sizes.append(x.size())
        x, idx1 = self.pool1(x)
        indices.append(idx1)

        x = self.enc_act2(self.enc_norm2(self.enc_conv2(x)))
        sizes.append(x.size())
        x, idx2 = self.pool2(x)
        indices.append(idx2)

        x = self.enc_act3(self.enc_norm3(self.enc_conv3(x)))
        sizes.append(x.size())
        x, idx3 = self.pool3(x)
        indices.append(idx3)
        x = x.flatten(start_dim=1)

        x = self.enc_fc(x)
        return x, indices, sizes

    # ----- Decoder -----
    def decoder(self, z, indices, sizes):
        idx1, idx2, idx3 = indices
        size1, size2, size3 = sizes

        x = self.dec_fc(z)
        x = x.view(size3[0], size3[1], -1)

        x = self.unpool3(x, idx3, output_size=size3)
        x = self.dec_act3(self.dec_deconv3(x))

        x = self.unpool2(x, idx2, output_size=size2)
        x = self.dec_act2(self.dec_deconv2(x))

        x = self.unpool1(x, idx1, output_size=size1)
        x = self.dec_tanh(self.dec_deconv1(x))

        return x

    # ----- Forward -----
    def forward(self, x):
        z, indices, sizes = self.encoder(x)
        x_r = self.decoder(z, indices, sizes)
        return x_r, z

In [17]:
class AudioAutoencoderUNet(nn.Module):
    """Un Autoencoder de Audio mejorado con conexiones skip (saltos) estilo U-Net.
    Las salidas del codificador se pasan directamente al decodificador.
    """
    def __init__(
        self,
        input_length=16000,
        bottleneck_size=64,
        kernel_size=5,
        channels=8,
        channel_mult=2,
        downsample=4,
    ):
        super().__init__()
        assert kernel_size % 2 == 1, "kernel_size should be odd for same padding"

        padding = kernel_size // 2

        # ----- Encoder (Estructura igual, solo cambia el forward para capturar los outputs) -----
        self.enc_conv1 = nn.Conv1d(1, channels, kernel_size=kernel_size, padding=padding)
        self.enc_norm1 = nn.BatchNorm1d(channels)
        self.enc_act1 = nn.PReLU()
        self.pool1 = nn.MaxPool1d(2, stride=downsample, return_indices=True)

        self.enc_conv2 = nn.Conv1d(channels, channels * channel_mult, kernel_size=kernel_size, padding=padding)
        self.enc_norm2 = nn.BatchNorm1d(channels * channel_mult)
        self.enc_act2 = nn.PReLU()
        self.pool2 = nn.MaxPool1d(2, stride=downsample, return_indices=True)

        self.enc_conv3 = nn.Conv1d(channels * channel_mult, channels * channel_mult ** 2, kernel_size=kernel_size, padding=padding)
        self.enc_norm3 = nn.BatchNorm1d(channels * channel_mult ** 2)
        self.enc_act3 = nn.PReLU()
        self.pool3 = nn.MaxPool1d(2, stride=downsample, return_indices=True)

        enc_out_len = input_length // downsample ** 3 * channels * channel_mult ** 2
        self.enc_fc = nn.Linear(enc_out_len, bottleneck_size)

        comp_rate = input_length / bottleneck_size
        print(f"Compression rate: {comp_rate:.1f}X")

        # ----- Decoder (Estructura igual, los saltos se implementan en el forward) -----
        self.dec_fc = nn.Linear(bottleneck_size, enc_out_len)

        self.unpool3 = nn.MaxUnpool1d(2, stride=downsample)
        # NOTA: No se necesita cambio de número de canales en Conv/Deconv para la suma (es una adición)
        self.dec_deconv3 = nn.Conv1d(channels * channel_mult ** 2, channels * channel_mult, kernel_size=kernel_size, padding=padding)
        self.dec_act3 = nn.PReLU()

        self.unpool2 = nn.MaxUnpool1d(2, stride=downsample)
        self.dec_deconv2 = nn.Conv1d(channels * channel_mult, channels, kernel_size=kernel_size, padding=padding)
        self.dec_act2 = nn.PReLU()

        self.unpool1 = nn.MaxUnpool1d(2, stride=downsample)
        self.dec_deconv1 = nn.Conv1d(channels, 1, kernel_size=kernel_size, padding=padding)
        self.dec_tanh = nn.Tanh()

    def encoder(self, x):
        sizes, indices, skip_outputs = [], [], []

        # Bloque 1
        x = self.enc_act1(self.enc_norm1(self.enc_conv1(x)))
        skip_outputs.append(x) # Guardamos la salida para el skip connection
        sizes.append(x.size())
        x, idx1 = self.pool1(x)
        indices.append(idx1)

        # Bloque 2
        x = self.enc_act2(self.enc_norm2(self.enc_conv2(x)))
        skip_outputs.append(x) # Guardamos la salida para el skip connection
        sizes.append(x.size())
        x, idx2 = self.pool2(x)
        indices.append(idx2)

        # Bloque 3
        x = self.enc_act3(self.enc_norm3(self.enc_conv3(x)))
        skip_outputs.append(x) # Guardamos la salida para el skip connection
        sizes.append(x.size())
        x, idx3 = self.pool3(x)
        indices.append(idx3)

        x = x.flatten(start_dim=1)
        x = self.enc_fc(x)

        return x, indices, sizes, skip_outputs

    def decoder(self, z, indices, sizes, skip_outputs):
        idx1, idx2, idx3 = indices
        size1, size2, size3 = sizes
        skip1, skip2, skip3 = skip_outputs # Skip connections en orden inverso (del final al inicio del encoder)

        x = self.dec_fc(z)
        x = x.view(size3[0], size3[1], -1)

        # Bloque 3 de Decodificación
        x = self.unpool3(x, idx3, output_size=size3)
        x = x + skip3 # APLICACIÓN DEL SKIP CONNECTION 3
        x = self.dec_act3(self.dec_deconv3(x))

        # Bloque 2 de Decodificación
        x = self.unpool2(x, idx2, output_size=size2)
        x = x + skip2 # APLICACIÓN DEL SKIP CONNECTION 2
        x = self.dec_act2(self.dec_deconv2(x))

        # Bloque 1 de Decodificación
        x = self.unpool1(x, idx1, output_size=size1)
        x = x + skip1 # APLICACIÓN DEL SKIP CONNECTION 1
        x = self.dec_tanh(self.dec_deconv1(x))

        return x

    def forward(self, x):
        # El encoder devuelve el bottleneck 'z', los 'indices', los 'sizes' y las 'skip_outputs'
        z, indices, sizes, skip_outputs = self.encoder(x)
        # El decoder recibe las 'skip_outputs' para la suma
        x_r = self.decoder(z, indices, sizes, skip_outputs)
        return x_r, z

### Model Base MSE

In [16]:
target_length = 16000 * TIME
bottleneck_size = 512 * TIME
downsample = 2
model_base = AudioAutoencoder(input_length=target_length, bottleneck_size=bottleneck_size, downsample=downsample)
model_base.to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model_base.parameters(), lr=1e-3)

n_epochs = 5
lr = 1e-3

Compression rate: 31.2X


In [17]:
model_base_history = train_autoencoder(model_base, train_loader, val_loader, device, n_epochs, lr, criterion)

Training: 100%|██████████| 5/5 [43:15<00:00, 519.07s/epoch, epoch=5, train loss=0.00457, val loss=0.00501]


In [63]:
model_base_history

{'train_loss': [np.float64(0.01582176488581408),
  np.float64(0.005910103290110954),
  np.float64(0.005102446212425319),
  np.float64(0.004780189662626506),
  np.float64(0.004569020154826247)],
 'val_loss': [np.float64(0.00685516271294783),
  np.float64(0.005198990182100754),
  np.float64(0.004731053738530887),
  np.float64(0.004413978136448271),
  np.float64(0.005007047248680306)]}

#### Test

In [53]:
batch = next(iter(test_loader))
test_x = batch[0].to(device)
x_r, _ = model_base(test_x)
x_r = x_r.detach().cpu()

In [54]:
Audio(test_x[4].squeeze().cpu().numpy(), rate=FS)

In [55]:
Audio(x_r[4].squeeze().cpu().numpy(), rate=FS)

### Model Base Perceptual Loss

In [15]:
def multi_scale_stft_loss(x, y, scales=[512, 1024, 2048], hop_ratio=0.25):
    """Simple implementation of the multi-scale STFT loss.
    https://arxiv.org/pdf/1910.11480
    """
    loss = 0
    for n_fft in scales:
        hop = int(n_fft * hop_ratio)
        window = torch.hann_window(n_fft, device=x.device)
        X = torch.stft(x.squeeze(), n_fft=n_fft, hop_length=hop, window=window, return_complex=True)
        Y = torch.stft(y.squeeze(), n_fft=n_fft, hop_length=hop, window=window, return_complex=True)
        loss += torch.mean(torch.abs(X.abs() - Y.abs()))
    return loss / len(scales)

In [26]:
target_length = 16000 * TIME
bottleneck_size = 512 * TIME
downsample = 2
model_base_perc = AudioAutoencoder(input_length=target_length, bottleneck_size=bottleneck_size, downsample=downsample)
model_base_perc.to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model_base_perc.parameters(), lr=1e-3)

# Reduced epochs because training is shorter and is enough
n_epochs = 2
lr = 1e-3

Compression rate: 31.2X


In [16]:
model_base_perc_history = train_autoencoder(model_base_perc, train_loader, val_loader, device, n_epochs, lr, criterion, multi_scale_stft_loss)

Training: 100%|██████████| 2/2 [17:29<00:00, 524.97s/epoch, epoch=2, train loss=0.45960, val loss=0.44386]


In [21]:
model_base_perc_history

{'train_loss': [np.float64(0.6383511093946603),
  np.float64(0.4596046091399641)],
 'val_loss': [np.float64(0.5064331277557041), np.float64(0.4438594724821008)]}

In [17]:
PATH = 'model_base_perc.pth'

# Use state_dict to save only the learned parameters
torch.save(model_base_perc.state_dict(), PATH)

print(f"Model weights saved to {PATH}")

Model weights saved to model_base_perc.pth


#### Test

In [18]:
batch = next(iter(test_loader))
x = batch[0].to(device)
x_r, _ = model_base_perc(x)
x_r = x_r.detach().cpu()

In [19]:
Audio(x[4].squeeze().cpu().numpy(), rate=FS)

In [20]:
Audio(x_r[4].squeeze().cpu().numpy(), rate=FS)

### Model U-Net Based with Perceptual Loss

In [18]:
target_length = 16000 * TIME
bottleneck_size = 512 * TIME
downsample = 2

model_unet = AudioAutoencoderUNet(input_length=target_length, bottleneck_size=bottleneck_size, downsample=downsample)
model_unet.to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model_unet.parameters(), lr=1e-3)

n_epochs = 2
lr = 1e-3

Compression rate: 31.2X


In [19]:
model_unet_history = train_autoencoder(model_unet, train_loader, val_loader, device, n_epochs, lr, criterion, multi_scale_stft_loss)

Training: 100%|██████████| 2/2 [17:13<00:00, 516.52s/epoch, epoch=2, train loss=2.27664, val loss=2.31395]


In [ ]:
PATH = 'model_unet.pth'

# Use state_dict to save only the learned parameters
torch.save(model_unet.state_dict(), PATH)

print(f"Model weights saved to {PATH}")

#### Test

In [21]:
batch = next(iter(test_loader))
x = batch[0].to(device)
x_r, _ = model_unet(x)
x_r = x_r.detach().cpu()

In [22]:
Audio(x[4].squeeze().cpu().numpy(), rate=FS)

In [23]:
Audio(x_r[4].squeeze().cpu().numpy(), rate=FS)

## Report

1. Data Strategy and Architectural Choices

To maximize perceptual reconstruction quality, the model was designed as a 1D Convolutional Autoencoder operating directly on the raw audio waveform (using the class baseline). This choice inherently solves the phase problem associated with spectral representations.

To manage the 30-second MTAT files, I employed a cropping strategy, using 3-second segments (48,000 samples at 16000 Hz) during training. This approach allowed me to maintain high network complexity while managing CUDA memory, which is preferable to reducing the model's layers or kernel count.

2. Experimental Results and Loss Function Analysis

I performed three experiments to evaluate the impact of loss function and architectural complexity on reconstruction quality, measured by validation MSE minimization.

Experiment A: Base Model with Mean Squared Error (MSE)

Training the base model with MSE achieved the lowest raw numerical loss on the validation set (0.00501), confirming good convergence. However, the resulting reconstructed audio suffered from significant audible background noise and artifacts. In the time domain, MSE minimizes amplitude differences but fails to enforce perceptually important spectral structure.

Experiment B: Base Model with Perceptual Loss

Switching to the Multi-Scale STFT Loss (MS-STFT) yielded a numerically higher validation loss (0.4439) because it is a different metric than MSE. Crucially, the perceptual quality improved dramatically. The MS-STFT loss forced the model to reconstruct audio with greater spectral fidelity and less audible noise, confirming that MS-STFT is superior for achieving good perceptual results, despite a higher numerical value compared to direct MSE optimization.

Experiment C: U-Net Architecture Test

To fulfill the requirement of improving reconstruction quality via architecture, I implemented a U-Net style Autoencoder using skip connections (addition). This model failed immediately. The resulting reconstruction was incoherent, and training was unstable, suggesting a fundamental structural flaw in implementing addition-based skip connections with the complex symmetry requirements of the MaxPool1d/MaxUnpool1d layers. This prevented me from verifying if the increased complexity would have further lowered the validation loss.

3. Conclusion

The experiments demonstrated that for raw audio autoencoders, perceptual loss is the dominant factor in achieving high-quality reconstruction. The Base Model trained with the Multi-Scale STFT Loss provided the best perceptual audio quality, even though the vanilla MSE model yielded the lowest raw validation error value. The structural improvement attempt via U-Net addition-based skip connections was unsuccessful, highlighting the difficulties in maintaining symmetry in deep 1D architectures.